# Ames Real Estate Valuation: Exploratory Data Analysis (EDA)

**Dataset**: Ames Housing Dataset (2,930 transactions, 80 features)  
**Objective**: Perform statistical exploration, outlier detection (>4,000 sq ft living area), feature distribution analysis, and correlation mapping.

---

## 1. Environment Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

dataset_path = Path('../data/AmesHousing.csv')
df_raw = pd.read_csv(dataset_path)
print(f"Raw Dataset Loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns.")
df_raw[['Overall Qual', 'Gr Liv Area', 'Garage Cars', 'Full Bath', 'Bedroom AbvGr', 'Year Built', 'SalePrice']].head()

## 2. Statistical Anomaly Removal & Data Cleaning

Per Ames Housing dataset literature (De Cock, 2011), properties with living area (`Gr Liv Area`) exceeding **4,000 sq ft** represent partial sales or extreme statistical outliers that distort linear regression slopes. We filter these anomalies along with missing target entries.

In [ ]:
base_features = ['Overall Qual', 'Gr Liv Area', 'Garage Cars', 'Full Bath', 'Bedroom AbvGr', 'Year Built']
all_needed_cols = base_features + ['SalePrice']

df_clean = df_raw[(df_raw['Gr Liv Area'] < 4000) & (df_raw['SalePrice'] > 0)].dropna(subset=all_needed_cols).copy()
print(f"Initial Record Count: {len(df_raw)}")
print(f"Cleaned Record Count: {len(df_clean)} (Removed {len(df_raw) - len(df_clean)} anomalies)")

print("\nTarget Variable ('SalePrice') Summary Statistics:")
print(df_clean['SalePrice'].describe().apply(lambda x: f"${x:,.2f}"))

## 3. Feature Engineering: Quality-to-Area Interaction

We introduce a domain-driven feature: **Quality-to-Area Interaction** (`Overall Qual` * `Gr Liv Area`). This captures non-linear price amplification where additional living area adds significantly more monetary value in high-quality luxury homes than in lower-grade structures.

In [ ]:
X = df_clean[base_features].copy()
qual_map = {'Very_Poor': 1, 'Poor': 2, 'Fair': 3, 'Below_Average': 4, 'Average': 5, 'Above_Average': 6, 'Good': 7, 'Very_Good': 8, 'Excellent': 9, 'Very_Excellent': 10}
if X['Overall Qual'].dtype == object:
    X['Overall Qual'] = X['Overall Qual'].map(qual_map).fillna(5)
X['Overall Qual'] = pd.to_numeric(X['Overall Qual'], errors='coerce').fillna(5).astype(float)
X['Qual_Area_Interaction'] = X['Overall Qual'] * X['Gr Liv Area']
y = df_clean['SalePrice']

print(f"Feature Matrix Shape: {X.shape}")
X.head()

## 4. Visualizing Feature Relationships & Heatmap

In [ ]:
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution plot
sns.histplot(y, kde=True, ax=axes[0], color='teal', bins=30)
axes[0].set_title('Property SalePrice Distribution ($)')

# Correlation Heatmap
full_df = X.copy()
full_df['SalePrice'] = y
sns.heatmap(full_df.corr(), annot=True, fmt='.2f', cmap='Blues', ax=axes[1])
axes[1].set_title('Feature Correlation Matrix')

plt.tight_layout()
plt.show()